## shape : combien de lignes et de colonnes ? Comparer à l'estimation de l'atelier 1

Il y a 800 lignes et 16 colonnes. Dans l'atelier 1 on avait trouvé 208 paniers, 194 produits et 208 utilisateurs avec l'API. Ici on retrouve bien les 208 paniers et les 208 utilisateurs, mais seulement 189 produits différents : 5 produits du catalogue ne sont dans aucun panier.

On a plus de lignes que de paniers car une ligne = un produit dans un panier (ligne de panier), pas une commande. Un panier contient en moyenne 3,85 produits (entre 2 et 6), d'où 800 lignes. Le même order_id apparaît donc plusieurs fois, c'est normal. Il ne faut pas additionner cart_total ligne par ligne car il est répété sur chaque ligne du panier.

## dtypes : les dates sont-elles reconnues comme des dates ?

Non, order_date est lue comme du texte (str). Il faut la convertir avec pd.to_datetime, et après elle est bien en datetime64. Les autres colonnes sont bien reconnues : les identifiants et quantity en int64, les prix, totaux et remises en float64.

## isna : quelles colonnes à compléter ou à supprimer ?

Une seule colonne a des valeurs manquantes : brand, avec 377 valeurs vides sur 800 (47 %). Ce n'est pas une erreur de collecte : certaines catégories n'ont jamais de marque dans l'API (groceries, home-decoration, kitchen-accessories, sports-accessories, tops, womens-dresses, womens-jewellery).

Il ne faut pas supprimer ces lignes (on perdrait presque la moitié des données) ni supprimer la colonne. On peut les remplacer par « Sans marque » si on fait une analyse par marque. Les 15 autres colonnes sont complètes.

## duplicated : combien de commandes en double ?

0 doublon exact. Mais si on regarde la clé métier cart_id + product_id, il y a 12 doublons : 12 paniers contiennent deux fois le même produit (24 lignes concernées). Par exemple CART-00007 contient deux fois le produit 56.

Dans 2 cas, les deux lignes sont identiques sauf order_date (même quantité, même prix) : ce sont de vrais doublons cachés par la date synthétique. Dans les 10 autres cas, les quantités sont différentes. Le nettoyage du notebook garde la première ligne, ce qui fait passer de 800 à 788 lignes, mais on perd un peu de montant (3 834 278,63 → 3 823 879,04). Il serait mieux d'additionner les quantités plutôt que de supprimer.

## describe : repérer les montants négatifs ou aberrants

Il n'y a pas de montant négatif ni de quantité nulle. Les quantités vont de 1 à 5, les prix unitaires de 0,79 à 36 999,99, les totaux de ligne de 0,99 à 184 999,95 et les remises de 0,04 % à 19,61 %.

Les valeurs les plus élevées viennent de la catégorie vehicle (Durango SXT RWD à 36 999,99, acheté 5 fois dans une ligne = 184 999,95). Ce n'est pas une erreur, c'est un produit cher, mais ça tire beaucoup la moyenne : le prix moyen est de 1 559 alors que la plupart des produits coûtent moins de 100. La catégorie vehicle représente à elle seule 1,88 million sur 3,83 millions de montant total. Il vaut mieux utiliser la médiane ou analyser par catégorie.

Les contrôles métier sont tous bons : quantité × prix = line_total (0 écart), total remisé ≤ total initial (0 anomalie), remise entre 0 et 100 (0 anomalie), et la somme des lignes d'un panier correspond bien à cart_total.

## Conclure : lister 3 problèmes de véracité à traiter

La date order_date est inventée par le script (tirée au hasard dans l'année 2025) et elle est générée ligne par ligne : 207 paniers sur 208 ont plusieurs dates différentes, alors qu'une commande devrait avoir une seule date. Il ne faut pas s'en servir pour une analyse dans le temps (saisonnalité, ventes par mois).

La marque (brand) manque dans 47 % des lignes (377 sur 800), car certaines catégories n'ont pas de marque dans l'API. Une analyse par marque serait donc faussée si on ne traite pas ces valeurs.

Des produits apparaissent deux fois dans le même panier (12 paniers). Ce n'est pas logique pour un panier, il faut regrouper les lignes (additionner les quantités) avant de charger les données dans MongoDB ou Spark.

En plus, les données restent fictives (DummyJSON) : un panier n'est pas une commande payée, donc ces chiffres ne représentent pas de vraies ventes.